### 20 Class Reduced to 4 Classes

***"... three
electrodes were positioned on the middle finger, four on the hand, six on the forearm, and
seven on the arm."***

Class 0 -> E1-E3 (Middle Finger)

Class 1 -> E4-E7 (Hand)

Class 2 -> E8-E13 (Forearm)

Class 3 -> E14-E20 (Arm)

In [1]:
import json 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nilearn.image import load_img, index_img, resample_to_img, new_img_like
from nilearn import plotting
from nilearn.maskers import NiftiMasker
from nilearn.datasets import fetch_atlas_destrieux_2009
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight


RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [2]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

ELECTRODE_TO_REGION = {
    electrode: region 
    for region, electrodes in REGION_TO_ELECTRODES.items() 
    for electrode in electrodes
}

REGION_TO_LABEL = {
    'Middle_Finger': 0,
    'Hand': 1,
    'Forearm': 2,
    'Arm': 3
}

LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

print("Region-to-Electrode Mapping:")
for region, electrodes in REGION_TO_ELECTRODES.items():
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {electrodes} ({len(electrodes)} electrodes)")

Region-to-Electrode Mapping:
  Middle_Finger (Class 0): ['E1', 'E2', 'E3'] (3 electrodes)
  Hand (Class 1): ['E4', 'E5', 'E6', 'E7'] (4 electrodes)
  Forearm (Class 2): ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'] (6 electrodes)
  Arm (Class 3): ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20'] (7 electrodes)


In [3]:
BIDS_ROOT = Path(r"../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"

subjects = ["sub-p0001"]
session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4


HRF_DELAY = 5.0
WINDOW = 1

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"Subject: {subjects[0]}")
print(f"Runs: {n_runs_per_subject}")
print(f"TR: {tr} s")
print(f"HRF delay: {HRF_DELAY} s")
print(f"Window: {WINDOW} volumes")

RESULTS_DIR = Path("results/4_Classes_ANN")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_LOG = RESULTS_DIR / "4_Classes_ANN_analysis_log.txt"
log_file = open(OUTPUT_LOG, 'w', encoding='utf-8')
experiments_results = []

Subject: sub-p0001
Runs: 4
TR: 2.0 s
HRF delay: 5.0 s
Window: 1 volumes


In [4]:
all_events = []
subject = subjects[0]
for run in range(1, n_runs_per_subject + 1):
    events_path = BIDS_ROOT / subject / session / "func" / f"{subject}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    df['subject'] = subject
    df['run'] = run
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()

#eletrode -> region mapping
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)

print(f"Subject: {subject}")
print(f"Total events: {len(events_df)}")
print(f"Stimulation events: {len(stim_events)}")
print(f"Unique electrodes: {stim_events['trial_type'].nunique()}")
print(f"Unique regions: {stim_events['region'].nunique()}")
print(f"\nSamples per run:")
print(stim_events.groupby('run').size().to_dict())
print(f"\nSamples per electrode:")
electrode_counts = stim_events['trial_type'].value_counts().sort_index()
print(f"  Min: {electrode_counts.min()}, Max: {electrode_counts.max()}, Mean: {electrode_counts.mean():.1f}")
print(f"\nSamples per region:")
region_counts = stim_events['region'].value_counts()
for region in ['Middle_Finger', 'Hand', 'Forearm', 'Arm']:
    count = region_counts.get(region, 0)
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {count} samples")

first_run_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii.gz")
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)
print(f"\nReference image shape: {first_run_img.shape}")

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
mask_data = np.isin(atlas_data, s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
print(f"Number of voxels in original S1 mask: {int(np.sum(mask_data))}")
print(f"Number of voxels in resampled S1 mask: {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)

print(f"\nS1 atlas indices: {s1_indices}")
print('Selected atlas regions:')
for i in s1_indices:
    print(f"  - {atlas.labels[i]}")

display = plotting.plot_roi(s1_mask_resampled, bg_img=ref_img, 
                            title='S1 ROI Mask (Left G_postcentral)',
                            display_mode='ortho', cut_coords=(0, -20, 60))
display.savefig(FIGURES_DIR / 's1_mask.png', dpi=150)
display.close()
print(f"\nS1 mask saved to: {FIGURES_DIR / 's1_mask.png'}")

Subject: sub-p0001
Total events: 328
Stimulation events: 160
Unique electrodes: 20
Unique regions: 4

Samples per run:
{1: 40, 2: 40, 3: 40, 4: 40}

Samples per electrode:
  Min: 8, Max: 8, Mean: 8.0

Samples per region:
  Middle_Finger (Class 0): 24 samples
  Hand (Class 1): 32 samples
  Forearm (Class 2): 48 samples
  Arm (Class 3): 56 samples

Reference image shape: (121, 144, 121, 250)
[fetch_atlas_destrieux_2009] Dataset found in C:\Users\duart\nilearn_data\destrieux_2009
Number of voxels in original S1 mask: 1138
Number of voxels in resampled S1 mask: 2187


C:\Users\duart\AppData\Local\Temp\ipykernel_14708\2782281552.py:38: UserWarning: 
The following regions are present in the atlas look-up table,
but missing from the atlas image:

 index          name
    42 L Medial_wall
   117 R Medial_wall

  atlas = fetch_atlas_destrieux_2009()
C:\Users\duart\AppData\Local\Temp\ipykernel_14708\2782281552.py:48: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  masker.fit(first_run_img)



S1 atlas indices: [28]
Selected atlas regions:
  - L G_postcentral

S1 mask saved to: results\4_Classes_ANN\figures\s1_mask.png


c:\Users\duart\miniconda3\envs\somatosensory_mapping\lib\site-packages\numpy\ma\core.py:2820: UserWarning: Warning: converting a masked element to nan.
  _data = np.array(data, dtype=dtype, copy=copy,


In [5]:
X_list = []
y_list = []
run_labels = []

for run in range(1, n_runs_per_subject + 1):
    bold_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii.gz")
    bold_img = load_img(str(bold_path))
    print(f"Run {run}...")
    run_events = stim_events[stim_events['run'] == run]
    for indx, (_, event) in enumerate(run_events.iterrows()):
        onset = event["onset"]
        trial_type = event["trial_type"]
        region = event["region"]
        target_time = onset + HRF_DELAY
        vol_indx = int(np.round(target_time / tr))
        
        if vol_indx < bold_img.shape[3]:
            vol_img = index_img(bold_img, vol_indx)
            vol_data = masker.transform(vol_img)
            if len(vol_data.shape) == 2:
                X_list.append(vol_data[0])
            else:
                X_list.append(vol_data)
            y_list.append(region) 
            run_labels.append(run)

X = np.array(X_list)
y = np.array(y_list)
run_labels = np.array(run_labels)
y_encoded = np.array([REGION_TO_LABEL[region] for region in y])
print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels shape: {y_encoded.shape}")
print(f"Number of unique regions: {len(np.unique(y_encoded))}")
print(f"\nLabel distribution:")
for region, label in REGION_TO_LABEL.items():
    count = np.sum(y_encoded == label)
    print(f"  {region} (Class {label}): {count} samples")

Run 1...
Run 2...
Run 3...
Run 4...

Feature matrix shape: (160, 2187)
Labels shape: (160,)
Number of unique regions: 4

Label distribution:
  Middle_Finger (Class 0): 24 samples
  Hand (Class 1): 32 samples
  Forearm (Class 2): 48 samples
  Arm (Class 3): 56 samples
